# Agent Memory in LangGraph

## First, Understand the Two Types of Memory

In [ ]:
SHORT-TERM MEMORY          LONG-TERM MEMORY
(Within a conversation)    (Across conversations)

"What did user say         "Who is this user?
 5 messages ago?"           What do they like?
                            What have we discussed
                            over past 6 months?"

Lives in RAM/context        Lives in Database
Dies when chat ends         Persists forever
Fast to access              Slower, needs retrieval

---

## The Core Problem: LLMs Are Stateless

Every time you call an LLM API, it remembers **nothing** from before:

In [ ]:
User (Monday):   "My name is Raj, I'm a ML engineer"
Bot:             "Nice to meet you Raj!"

User (Tuesday):  "What's my name?"
Bot:             "I don't know your name"  ← PROBLEM
                  (stateless — remembered nothing)

Your application code must **manually manage memory** and inject it into every prompt. The LLM itself stores nothing.

---

## The Full Memory Architecture

In [ ]:
User Message
     │
     ▼
┌─────────────────────────────────────────┐
│           MEMORY MANAGER               │
│                                         │
│  1. Retrieve relevant long-term memory  │
│  2. Load current short-term context     │
│  3. Build final prompt                  │
│  4. After response → update memories   │
└─────────────────────────────────────────┘
     │
     ▼
┌──────────┐    ┌─────────────────────────┐
│   LLM    │    │      STORAGE LAYER      │
│  (GPT/   │    │                         │
│  Claude) │    │  Short-term: Redis      │
└──────────┘    │  Long-term:  PostgreSQL │
                │  Semantic:   Pinecone   │
                └─────────────────────────┘

---

## Part 1: Short-Term Memory

Short-term memory = **the current conversation window**. What was said in this session.

### What to Store

In [ ]:
Session data:
{
  session_id:   "sess_abc123",
  user_id:      "user_456",
  started_at:   "2026-02-25T10:00:00",
  messages: [
    {"role": "user",      "content": "Hello, I need help with Python"},
    {"role": "assistant", "content": "Sure! What specifically?"},
    {"role": "user",      "content": "How do I read a CSV?"},
    {"role": "assistant", "content": "Use pandas: pd.read_csv(...)"},
  ]
}

### Where to Store: Redis

Redis is the industry standard for short-term chat memory because:

In [ ]:
✅ Stored in RAM → microsecond read/write
✅ Built-in TTL (auto-expires after X hours)
✅ Simple key-value structure perfect for sessions
✅ Handles thousands of concurrent sessions easily

In [ ]:
import redis
import json

r = redis.Redis(host='localhost', port=6379)

# Save message to session
def save_message(session_id, role, content):
    key = f"session:{session_id}:messages"
    message = {"role": role, "content": content}
    r.rpush(key, json.dumps(message))   # append to list
    r.expire(key, 3600)                 # expire in 1 hour

# Load full conversation
def load_conversation(session_id):
    key = f"session:{session_id}:messages"
    messages = r.lrange(key, 0, -1)    # get all messages
    return [json.loads(m) for m in messages]

# Usage:
save_message("sess_abc123", "user", "How do I read a CSV?")
save_message("sess_abc123", "assistant", "Use pd.read_csv()")

history = load_conversation("sess_abc123")
# → [{"role": "user", "content": "How do I read a CSV?"},
#    {"role": "assistant", "content": "Use pd.read_csv()"}]

### The Context Window Problem

LLMs have a limited context window (e.g., 8K, 128K tokens). Long conversations overflow it:

In [ ]:
Context window: 8000 tokens

Message 1:   150 tokens
Message 2:   200 tokens
...
Message 40:  ← OVERFLOW! Can't fit all 40 messages

**Solutions:**

#### Strategy 1 — Sliding Window (Simplest)

In [ ]:
def get_context_window(session_id, max_messages=20):
    all_messages = load_conversation(session_id)
    # Always keep system prompt + last N messages
    return all_messages[-max_messages:]

In [ ]:
Full history:    [msg1, msg2, msg3, ... msg50]
Sent to LLM:                         [msg31 ... msg50]
                                      ↑ only last 20

Downside: model forgets what was said early in conversation.

#### Strategy 2 — Summarization (Smarter)

`Keep last 'n' conversation as is and summarize earlier message.`

In [ ]:
Before:
  [msg1, msg2, ... msg40, msg41, msg42, msg43, msg44, msg45]

After compression:
  [SUMMARY of msg1-msg40,  msg41, msg42, msg43, msg44, msg45]
   ↑ 1 message             ↑ last 5 messages in full

This is what ChatGPT does internally for long conversations.

In [ ]:
def compress_old_messages(messages, keep_recent=10):
    if len(messages) <= keep_recent:
        return messages

    old_messages = messages[:-keep_recent]
    recent_messages = messages[-keep_recent:]

    # Ask LLM to summarize old messages
    summary = llm.call(
        f"Summarize this conversation in 3 sentences: {old_messages}"
    )

    return [
        {"role": "system", "content": f"Earlier summary: {summary}"},
        *recent_messages   # keep recent messages in full
    ]

---

## Part 2: Long-Term Memory

Long-term memory persists **across sessions** — it's what makes the bot feel like it truly knows the user.

### Three Types of Long-Term Memory

In [ ]:
1. EPISODIC MEMORY       2. SEMANTIC MEMORY       3. PREFERENCE MEMORY
"What happened"          "Facts about user"        "What user likes/dislikes"

"Raj asked about         "Raj is a ML engineer     "Raj prefers code examples
 LoRA last week"          at startup in Chennai"    over theory explanations"
 "Raj was frustrated      "Raj uses Python 3.11"   "Raj likes short answers"
  with deployment"        "Raj has 5yr experience"

### Where to Store: PostgreSQL

In [ ]:
-- Users table
CREATE TABLE users (
    user_id      UUID PRIMARY KEY,
    created_at   TIMESTAMP DEFAULT NOW()
);

-- Long-term facts/preferences
CREATE TABLE user_memory (
    id           UUID PRIMARY KEY,
    user_id      UUID REFERENCES users(user_id),
    memory_type  VARCHAR(20),  -- 'fact', 'preference', 'episode'
    content      TEXT,
    importance   FLOAT,        -- 0.0 to 1.0, how important is this
    created_at   TIMESTAMP DEFAULT NOW(),
    last_used_at TIMESTAMP,
    access_count INT DEFAULT 0
);

-- Conversation sessions
CREATE TABLE sessions (
    session_id   UUID PRIMARY KEY,
    user_id      UUID REFERENCES users(user_id),
    summary      TEXT,         -- LLM-generated summary
    started_at   TIMESTAMP,
    ended_at     TIMESTAMP
);

### How to Extract and Save Memories

After every conversation, run an **extraction step**:

In [ ]:
def extract_memories_from_conversation(user_id, conversation):

    # Ask LLM to extract important facts
    extraction_prompt = f"""
    Analyze this conversation and extract:
    1. Important facts about the user (name, job, location, skills)
    2. User preferences (communication style, detail level)
    3. Problems they faced
    4. Decisions they made

    Return as JSON.

    Conversation: {conversation}
    """

    extracted = llm.call(extraction_prompt)
    # Returns:
    # {
    #   "facts": ["User is a ML engineer", "Uses Python"],
    #   "preferences": ["Prefers code examples", "Likes brevity"],
    #   "episodes": ["Struggled with LoRA deployment today"]
    # }

    # Save each memory to PostgreSQL
    for fact in extracted["facts"]:
        save_memory(user_id, "fact", fact, importance=0.9)

    for pref in extracted["preferences"]:
        save_memory(user_id, "preference", pref, importance=0.8)

    for episode in extracted["episodes"]:
        save_memory(user_id, "episode", episode, importance=0.6)

---

## Part 3: Semantic Memory (Vector Search)

For large memory stores, you can't inject ALL memories into every prompt. You need to **find the relevant ones** based on what the user is currently asking.

This is where **vector databases** come in:

In [ ]:
User asks: "Help me optimize my training loop"

Without semantic search:
  Load ALL 500 memories about this user
  → too many tokens, too slow

With semantic search:
  Search memories similar to "training loop optimization"
  → Returns only: ["User does ML training", "User uses PyTorch",
                   "User had memory issues with batch size 128"]
  → 3 relevant memories instead of 500!

### Vector Database Architecture

In [ ]:
from sentence_transformers import SentenceTransformer
import pinecone  # or Qdrant, Weaviate, pgvector

encoder = SentenceTransformer('all-MiniLM-L6-v2')

# When saving a memory:
def save_memory_with_embedding(user_id, content, memory_type):

    # Convert memory to vector
    vector = encoder.encode(content).tolist()

    # Save to vector DB with metadata
    pinecone_index.upsert([{
        "id":       f"{user_id}_{hash(content)}",
        "values":   vector,
        "metadata": {
            "user_id":     user_id,
            "content":     content,
            "memory_type": memory_type,
            "created_at":  datetime.now().isoformat()
        }
    }])

# When retrieving relevant memories:
def get_relevant_memories(user_id, current_message, top_k=5):

    # Convert current message to vector
    query_vector = encoder.encode(current_message).tolist()

    # Find most similar memories
    results = pinecone_index.query(
        vector=query_vector,
        filter={"user_id": user_id},   # only this user's memories
        top_k=top_k,
        include_metadata=True
    )

    return [r["metadata"]["content"] for r in results["matches"]]

---

## Part 4: Building the Final Prompt

All memory types come together when building the prompt sent to the LLM:

In [ ]:
def build_prompt(user_id, session_id, user_message):

    # 1. Load user profile facts (always include)
    user_facts = get_user_facts(user_id)

    # 2. Retrieve relevant long-term memories
    relevant_memories = get_relevant_memories(user_id, user_message)

    # 3. Load short-term conversation history
    recent_conversation = load_conversation(session_id)

    # 4. Assemble final prompt
    system_prompt = f"""
    You are a helpful assistant.

    === USER PROFILE ===
    {user_facts}
    (e.g., "Raj, ML Engineer, Chennai, 5yr experience, Python expert")

    === RELEVANT MEMORIES ===
    {relevant_memories}
    (e.g., "Prefers code examples", "Had deployment issues last week")

    === CURRENT CONVERSATION ===
    {recent_conversation}
    """

    return [
        {"role": "system",  "content": system_prompt},
        {"role": "user",    "content": user_message}
    ]

The assembled prompt looks like this:

In [ ]:
SYSTEM:
  You are a helpful assistant.

  User Profile:
    - Name: Raj
    - Role: ML Engineer at startup
    - Location: Chennai
    - Experience: 5 years Python

  Relevant Memories:
    - Prefers code examples over theory
    - Had GPU memory issues last week with batch_size=128
    - Previously asked about LoRA fine-tuning

  Recent Conversation:
    User: "My training loop is slow"
    Bot:  "What's your current batch size?"
    User: "256"

USER:
  "Should I use gradient accumulation?"

Now the LLM can give a **perfectly personalized** answer — knowing the user's background, their past problems, their preferences, and the current conversation.

---

## Part 5: Memory Lifecycle Management

In production, memories need to be **maintained** or they become stale and bloated:

### Importance Decay

In [ ]:
# Memories fade over time if not accessed
def update_memory_importance(memory_id):
    memory = get_memory(memory_id)

    days_since_used = (now - memory.last_used_at).days

    # Decay formula: importance drops over time
    new_importance = memory.importance * (0.99 ** days_since_used)

    update_memory(memory_id, importance=new_importance)

# Prune memories below threshold
def prune_old_memories(user_id, threshold=0.1):
    delete_memories_where(
        user_id=user_id,
        importance_below=threshold
    )

### Memory Conflict Resolution

In [ ]:
# What if a fact changes?
# Old memory: "Raj works at startup X"
# New memory: "Raj joined company Y"

def upsert_memory(user_id, new_content, memory_type):
    # Ask LLM: does this contradict any existing memory?
    existing = get_memories(user_id, memory_type)

    conflict_check = llm.call(f"""
    New information: "{new_content}"
    Existing memories: {existing}
    Does the new info contradict any existing memory?
    If yes, which one should be deleted?
    """)

    if conflict_check["has_conflict"]:
        delete_memory(conflict_check["outdated_memory_id"])

    save_memory(user_id, new_content, memory_type)

---

## The Full Production Architecture

In [ ]:
                        USER MESSAGE
                             │
                             ▼
                    ┌─────────────────┐
                    │  API Gateway    │
                    │  (rate limit,   │
                    │   auth)         │
                    └────────┬────────┘
                             │
                             ▼
                    ┌─────────────────┐
                    │  MEMORY MANAGER │◄──────────────────┐
                    │                 │                    │
                    │ 1. Get profile  │                    │
                    │ 2. Semantic     │                    │
                    │    retrieval    │                    │
                    │ 3. Load context │                    │
                    │ 4. Build prompt │                    │
                    └────────┬────────┘                   │
                             │                            │
              ┌──────────────┼──────────────┐             │
              ▼              ▼              ▼             │
       ┌──────────┐  ┌──────────────┐  ┌──────────┐      │
       │  REDIS   │  │  POSTGRESQL  │  │ PINECONE │      │
       │          │  │              │  │ (Vector  │      │
       │Short-term│  │ Long-term    │  │  Search) │      │
       │ sessions │  │ facts, prefs │  │          │      │
       │ (1hr TTL)│  │ episodes     │  │ Semantic │      │
       └──────────┘  └──────────────┘  │ retrieval│      │
                                       └──────────┘      │
                             │                            │
                             ▼                            │
                    ┌─────────────────┐                   │
                    │    LLM API      │                   │
                    │  (GPT/Claude)   │                   │
                    └────────┬────────┘                   │
                             │                            │
                             ▼                            │
                    ┌─────────────────┐                   │
                    │ POST-PROCESSING │                   │
                    │                 │                   │
                    │ 1. Save to      │                   │
                    │    session      │                   │
                    │ 2. Extract &    │───────────────────┘
                    │    save new     │ (update memories
                    │    memories     │  after every turn)
                    └────────┬────────┘
                             │
                             ▼
                        RESPONSE TO USER

---

## Quick Technology Choices Summary

| Need | Tool | Why |

|---|---|---|

| Short-term session | **Redis** | In-memory, fast, auto-expiry |

| Long-term facts | **PostgreSQL** | Reliable, queryable, relational |

| Semantic search | **Pinecone / Qdrant / pgvector** | Find relevant memories by meaning |

| Context compression | **LLM summarization** | Fit long chats into context window |

| Embeddings | **sentence-transformers** | Convert memories to searchable vectors |

---

## Summary: The 3-Layer Memory Stack

In [ ]:
LAYER 1 — SHORT TERM (Redis):
  What: Current conversation messages
  When: Every message, within one session
  Expires: 1-24 hours after session ends
  Injected: Always — as conversation history

LAYER 2 — LONG TERM (PostgreSQL):
  What: User facts, preferences, past episodes
  When: Extracted after each session ends
  Expires: Slowly decays by importance score
  Injected: Always — as user profile section

LAYER 3 — SEMANTIC (Vector DB):
  What: All memories, vectorized
  When: Saved alongside PostgreSQL entries
  Expires: Same as PostgreSQL
  Injected: Selectively — only top-K most
             relevant to current message

> **One-liner:** Short-term memory is a **notepad** (Redis) you write on during conversation and throw away after. Long-term memory is a **profile database** (PostgreSQL) you keep forever. Semantic memory is a **smart search engine** (Pinecone) that finds which long-term memories are relevant right now. All three are injected into the prompt together before calling the LLM.

---

# Async Memory Persistence — Respond First, Persist Later

Memory writes (Redis, PostgreSQL, Pinecone) add latency **after** the LLM responds.

`The fix: send the response immediately, persist in the background.`

In [ ]:
REQUEST → PRE-FETCH memory (blocking) → LLM CALL (blocking) → SEND RESPONSE
                                                                      │
                                                               PERSIST ASYNC ──► Redis / PostgreSQL / Pinecone

> **Rule:** Reads are always blocking (needed to build the prompt). Writes are always async (client doesn't need to wait).

---

## Option 1 — FastAPI BackgroundTasks *(simple, recommended)*

Best for most production apps. FastAPI sends the HTTP response first, then runs the background function.

In [ ]:
from fastapi import FastAPI, BackgroundTasks

app = FastAPI()

@app.post("/chat")
async def chat(request: ChatRequest, background_tasks: BackgroundTasks):

    # PRE-FETCH: blocking reads needed to build the prompt
    session_msgs     = await redis.lrange(f"session:{request.session_id}", 0, -1)
    relevant_memories = await vector_db.search(request.message, user_id=request.user_id)

    prompt   = build_prompt(session_msgs, relevant_memories, request.message)
    response = await llm.ainvoke(prompt)                   # LLM call

    # Schedule persistence — runs AFTER response is returned to client
    background_tasks.add_task(
        persist_memory,
        request.user_id, request.session_id,
        request.message, response.content
    )

    return {"response": response.content}                  # ← client gets this NOW


async def persist_memory(user_id, session_id, user_msg, bot_msg):
    # Save conversation turn to Redis
    await redis.rpush(f"session:{session_id}",
                      json.dumps({"role": "user",      "content": user_msg}),
                      json.dumps({"role": "assistant",  "content": bot_msg}))
    await redis.expire(f"session:{session_id}", 3600)

    # Extract and save long-term memories to PostgreSQL + Pinecone
    memories = await extract_memories_no_pii(user_msg, bot_msg)
    await save_long_term_memories(user_id, memories)

**Tradeoff:** If the server crashes between response and persistence, that turn is lost. Acceptable for most apps.

---

## Option 2 — Redis Message Queue *(production, high reliability)*

Decouples the API server from persistence entirely. A separate worker process consumes jobs from a queue — survives server restarts, scales independently.

In [ ]:
# ── API SERVER (producer) ──────────────────────────────────────────────────
import redis.asyncio as aioredis, json, time

r = aioredis.Redis()

@app.post("/chat")
async def chat(request: ChatRequest):

    # PRE-FETCH (blocking reads)
    session_msgs      = await r.lrange(f"session:{request.session_id}", 0, -1)
    relevant_memories = await vector_db.search(request.message, user_id=request.user_id)

    prompt   = build_prompt(session_msgs, relevant_memories, request.message)
    response = await llm.ainvoke(prompt)

    # Push job to queue — non-blocking, ~1ms
    await r.rpush("memory_queue", json.dumps({
        "user_id":    request.user_id,
        "session_id": request.session_id,
        "user_msg":   request.message,
        "bot_msg":    response.content,
        "ts":         time.time()
    }))

    return {"response": response.content}                  # ← returns immediately

In [ ]:
# ── WORKER PROCESS (consumer) — run separately: python worker.py ──────────
async def memory_worker():
    while True:
        _, raw = await r.blpop("memory_queue", timeout=0)  # blocks until job arrives
        job    = json.loads(raw)

        await redis.rpush(f"session:{job['session_id']}",
                          json.dumps({"role": "user",     "content": job["user_msg"]}),
                          json.dumps({"role": "assistant", "content": job["bot_msg"]}))
        await redis.expire(f"session:{job['session_id']}", 3600)

        memories = await extract_memories_no_pii(job["user_msg"], job["bot_msg"])
        await save_long_term_memories(job["user_id"], memories)

asyncio.run(memory_worker())

In [ ]:
API Server ──rpush──► Redis Queue ──blpop──► Worker Process
(~1ms push)           (durable buffer)       (persists reliably)

**Tradeoff:** Slightly more infrastructure. Worth it at scale — workers can be scaled independently, and jobs survive API restarts.

---

## When to Use Which

| | **BackgroundTasks** | **Redis Queue** |

|---|---|---|

| **Setup** | Zero — built into FastAPI | Worker process + queue |

| **Reliability** | Lost if server crashes mid-task | Durable — survives crashes |

| **Scale** | Tied to API server | Workers scale independently |

| **Use when** | Low–medium traffic, simple apps | High traffic, production systems |

# Protecting PII in ChatBot Memory

## What Is PII?

PII = Personally Identifiable Information — any data that can identify a specific person.

In [ ]:
DIRECT PII (obviously sensitive)      INDIRECT PII (sensitive in combination)
──────────────────────────────────    ───────────────────────────────────────
Names, Email, Phone                   City + Job + Age  → can identify someone
Aadhaar / SSN / Passport              IP Address        → traces to a person
Credit Card numbers                   Medical info:     "I have diabetes"
Date of Birth                         Financial info:   "My salary is ₹20 LPA"

In [ ]:
**Why it matters in chatbots:**
User: "My credit card 4532-1234-5678-9010 was charged wrongly"

Without protection → CC number saved to Redis, stored in PostgreSQL,
                      embedded in vector DB, injected into future prompts,
                      and appears in logs.  ← Multiple breach points!

---

## The 4-Layer Defense

In [ ]:
User Message → [LAYER 1: Detect PII] → [LAYER 2: Anonymize] → [LAYER 3: Safe Storage] → [LAYER 4: Audit]

---

## Layer 1: PII Detection

In [ ]:
### Approach A — Regex (structured PII)
import re

PII_PATTERNS = {
    "email":       r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b',
    "phone":       r'\b[6-9]\d{9}\b',
    "credit_card": r'\b\d{4}[\s\-]?\d{4}[\s\-]?\d{4}[\s\-]?\d{4}\b',
    "aadhaar":     r'\b\d{4}\s?\d{4}\s?\d{4}\b',
    "ssn_us":      r'\b\d{3}-\d{2}-\d{4}\b',
}

def detect_pii_regex(text):
    found = []
    for pii_type, pattern in PII_PATTERNS.items():
        for match in re.finditer(pattern, text, re.IGNORECASE):
            found.append({"type": pii_type, "value": match.group(),
                          "start": match.start(), "end": match.end()})
    return found

In [ ]:
### Approach B — NER Model (names, locations)
import spacy
nlp = spacy.load("en_core_web_lg")

def detect_pii_ner(text):
    doc = nlp(text)
    return [{"type": ent.label_, "value": ent.text,
             "start": ent.start_char, "end": ent.end_char}
            for ent in doc.ents
            if ent.label_ in {"PERSON", "GPE", "DATE", "MONEY"}]
# "I'm Raj Kumar from Chennai" → PERSON: "Raj Kumar", GPE: "Chennai"

In [ ]:
### Approach C — LLM (contextual / custom IDs)
def detect_pii_llm(text):
    prompt = f"""Identify ALL PII in this text. Return JSON:
    {{"has_pii": true/false, "pii_items": [{{"type": "...", "value": "...", "risk": "high/medium/low"}}]}}
    Text: "{text}" """
    return llm.call(prompt, response_format="json")
# Catches: "My employee ID is EMP-4521" ← missed by regex and NER

> **Best Practice**: Use all three — Regex for structure, NER for language, LLM for context.

---

## Layer 2: Anonymization

In [ ]:
### Strategy A — Redaction (delete PII)
def redact_pii(text, pii_items):
    for pii in sorted(pii_items, key=lambda x: x["start"], reverse=True):
        placeholder = f"[{pii['type'].upper()}_REDACTED]"
        text = text[:pii["start"]] + placeholder + text[pii["end"]:]
    return text

# "Hi I'm Raj, card 4532-1234-5678-9010"
# → "Hi I'm [PERSON_REDACTED], card [CREDIT_CARD_REDACTED]"

In [ ]:
### Strategy B — Tokenization (replace + restore later)
import uuid, redis
r = redis.Redis()

def tokenize_pii(text, pii_items, session_id):
    token_map = {}
    for pii in sorted(pii_items, key=lambda x: x["start"], reverse=True):
        token = f"[TOKEN_{pii['type'].upper()}_{uuid.uuid4().hex[:8]}]"
        r.setex(f"pii_token:{session_id}:{token}", 3600, pii["value"])
        token_map[token] = pii["value"]
        text = text[:pii["start"]] + token + text[pii["end"]:]
    return text, token_map

# Use when you need to restore PII later (e.g., process a payment)
# Token stored in Redis with 1hr TTL — real value never hits memory/DB

In [ ]:
### Strategy C — Generalization (abstract the value)
# Instead of saving: "User's name is Raj Kumar"
# Save:              "User mentioned their name"
# Fact preserved, actual PII is not.

---

## Layer 3: Safe Storage Rules

In [ ]:
USER MESSAGE: "I'm Raj, Aadhaar 1234-5678-9012, help with tax filing FY2025"

  Redis (session):       ✅ "User [PERSON_REDACTED] needs tax filing help"
                         ❌ Name, Aadhaar number

  PostgreSQL (long-term):✅ "User has tax filing questions for FY2025"
                         ❌ Real name, any ID numbers

  Logs:                  ✅ Request ID, timestamp, latency, status code
                         ❌ Message content, any user data

In [ ]:
**Encrypt unavoidable PII at rest:**
from cryptography.fernet import Fernet
cipher = Fernet(Fernet.generate_key())  # store key in KMS, not in code!

def save_user_email(user_id, email):
    db.execute("INSERT INTO users (user_id, email_enc) VALUES (?, ?)",
               (user_id, cipher.encrypt(email.encode())))

**Isolate PII in a separate DB table** with strict access controls, separate backup policy, and row-level security.

---

## Layer 4: Audit & Monitoring

In [ ]:
def audit_memories_for_pii(user_id):
    for memory in get_all_memories(user_id):
        pii_found = detect_pii_regex(memory["content"])
        if pii_found:
            clean = redact_pii(memory["content"], pii_found)
            update_memory(memory["id"], content=clean)
            alert_security_team(memory["id"], pii_found)

# Run as a daily cron job + on every memory save as safety net

---

## The Full Safeguarded Pipeline

In [ ]:
def process_user_message(user_id, session_id, raw_message):
    pii_found   = detect_all_pii(raw_message)           # 1. Detect

    if pii_found:
        high_risk = [p for p in pii_found
                     if p["type"] in ["credit_card", "ssn", "aadhaar"]]
        if high_risk:
            safe_message = redact_pii(raw_message, pii_found)   # 2a. Redact hard PII
        else:
            safe_message, _ = tokenize_pii(raw_message,         # 2b. Tokenize soft PII
                                           pii_found, session_id)
    else:
        safe_message = raw_message

    save_message(session_id, "user", safe_message)      # 3. Save safe version only
    prompt   = build_prompt(user_id, session_id, safe_message)  # 4. Build prompt
    response = llm.call(prompt)                         # 5. Call LLM

    resp_pii = detect_pii_regex(response)               # 6. Scan response too
    if resp_pii:
        response = redact_pii(response, resp_pii)       #    LLM may echo PII back!

    save_message(session_id, "assistant", response)     # 7. Save clean response
    save_long_term_memories(user_id,                    # 8. Extract PII-free memories
                            extract_memories_no_pii(safe_message, response))
    return response

---

## Compliance: GDPR / India DPDP Act

| Requirement | Implementation |

|---|---|

| **Right to erasure** | Delete user from Redis, PostgreSQL, Pinecone on request |

| **Data minimization** | Only store what the chatbot strictly needs |

| **Storage limitation** | TTL on all memory, auto-delete after inactivity |

| **Purpose limitation** | Memory system only serves the chatbot, not ads/analytics |

In [ ]:
def delete_all_user_data(user_id):
    redis_client.delete(f"user:{user_id}:*")
    postgres.execute("DELETE FROM user_memory WHERE user_id = %s", user_id)
    postgres.execute("DELETE FROM user_pii WHERE user_id = %s", user_id)
    pinecone.delete(filter={"user_id": user_id})
    audit_log(f"Full erasure for user {user_id}")

---

## Summary Checklist

In [ ]:
DETECTION       ✅ Regex (cards, phones, emails)  ✅ NER (names, locations)  ✅ LLM (context)
BEFORE SAVING   ✅ Redact high-risk PII            ✅ Tokenize restorable PII ✅ Generalize the rest
STORAGE         ✅ Encrypt PII at rest             ✅ Separate PII table      ✅ TTL on sessions
IN TRANSIT      ✅ Never log raw messages           ✅ Scan LLM responses      ✅ HTTPS everywhere
MONITORING      ✅ Daily PII audit on memories      ✅ Alert on leaks          ✅ Erasure endpoint

> **One-liner:** Treat PII like radioactive material — detect it the moment it enters, immediately neutralize it (redact/tokenize/generalize), store only the safe version, encrypt what you must keep, and have a complete destruction procedure ready when the user asks to be forgotten.

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
import os

load_dotenv()

api_key = os.environ['UNIFIED_LLM_KEY']
# print(api_key)
base_url = ""

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    max_tokens=256,
    api_key=api_key,
    base_url=base_url
)

# Short-Term Memory in LangGraph

**Short-term memory** stores conversation state during active sessions using **checkpointers**.

---

## Key Concepts

| **Concept** | **Explanation** |

|-------------|-----------------|

| **Checkpointer** | Saves state after each node execution |

| **thread_id** | Unique ID to isolate conversations |

| **Annotated[list, add]** | Accumulates messages (not replaces) |

| **MemorySaver** | In-memory (dev), lost on restart |

| **SqliteSaver** | Persistent (prod), survives restart |

---

## How It Works

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, Annotated
from operator import add

# State with message accumulation
class ChatState(TypedDict):
    messages: Annotated[list[AnyMessage], add]  # Appends, not replaces

# Compile with checkpointer
graph = workflow.compile(checkpointer=MemorySaver())

# Same thread_id = Same conversation
config = {"configurable": {"thread_id": "user-1"}}

# Call 1: State = ["Hi, I'm Bob", "Hello Bob!"]
graph.invoke({"messages": [HumanMessage("Hi, I'm Bob")]}, config)

# Call 2: State = ["Hi, I'm Bob", "Hello Bob!", "What's my name?", "Your name is Bob"]
graph.invoke({"messages": [HumanMessage("What's my name?")]}, config)

---

## State Accumulation

In [ ]:
**❌ Without `add`** (replaces):
messages: list[AnyMessage]  # Each invoke replaces entire list

In [ ]:
**✅ With `add`** (accumulates):
messages: Annotated[list[AnyMessage], add]  # Appends to existing list

---

## Check State

In [ ]:
# Get current state
state = graph.get_state(config)
print(state.values["messages"])  # All messages

# Get history
history = graph.get_state_history(config)
for checkpoint in history:
    print(checkpoint.values)  # State at each step

---

## Memory Types

In [ ]:
# Development: In-memory (lost on restart)
from langgraph.checkpoint.memory import MemorySaver
checkpointer = MemorySaver()

# Production: Persistent (survives restart)
from langgraph.checkpoint.sqlite import SqliteSaver
checkpointer = SqliteSaver.from_conn_string("memory.db")

---

## Best Practices

✅ Use `Annotated[list, add]` to accumulate messages  

✅ Unique `thread_id` per user/conversation  

✅ Limit messages (e.g., last 10) to prevent unbounded growth  

✅ MemorySaver for dev, SqliteSaver for production

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import HumanMessage, AIMessage, AnyMessage
from operator import add

# Define state schema
class ChatState(TypedDict):
    messages: Annotated[list[AnyMessage], add]

# Define nodes
def chatbot_node(state: ChatState):
    """Main chatbot that responds to user messages"""
    user_message = state["messages"][-1].content
    
    # Simple response logic with memory
    response = llm.invoke(state["messages"])
    
    return {"messages": [response]}

def log_node(state: ChatState):
    """Log conversation for analytics"""
    print(f"📝 Total messages: {len(state['messages'])}")
    return {}

# Build graph
workflow = StateGraph(ChatState)

# Add nodes
workflow.add_node("chatbot", chatbot_node)
workflow.add_node("logger", log_node)

# Add edges
workflow.add_edge(START, "chatbot")
workflow.add_edge("chatbot", "logger")
workflow.add_edge("logger", END)

# Compile with checkpointer
checkpointer = MemorySaver()
graph = workflow.compile(checkpointer=checkpointer)

# First interaction
result1 = graph.invoke(
    {"messages": [HumanMessage(content="hi! i am Bob")]},
    {"configurable": {"thread_id": "1"}}
)

# print("Bot:", result1["messages"][-1].content)
print("Bot:", [result.content for result in result1["messages"]])

# Second interaction - remembers Bob
result2 = graph.invoke(
    {"messages": [HumanMessage(content="What's my name?")]},
    {"configurable": {"thread_id": "1"}}
)

print("Bot:", [result.content for result in result2["messages"]])

## Checkpointer in SubGraph

- If your graph contains subgraphs, you only need to `provide the checkpointer when compiling the parent graph. LangGraph will automatically propagate the checkpointer to the child subgraphs.`

**Subgraph Structure:**

- **Intent Classification Subgraph with its own START and END**

    - classify node: Detects intent (name_query, weather_query, general)

    - validate node: Validates the intent

    - Has its own END node (returns to parent)

**Parent Graph:**

- Calls the subgraph as a node

- chatbot node: Generates LLM response

- logger node: Logs conversation state

**Memory Propagation:**

- Checkpointer defined ONLY on parent graph

- Automatically propagates to subgraph

- Both parent and subgraph share the same state

- thread_id preserves memory across invocations

**Test Flow:**

1. **Conversation 1**: "Hi! I'm Alice" → Subgraph classifies intent → Chatbot responds

2. **Conversation 2**: "What's my name?" → Uses same thread_id → Remembers Alice from previous conversation

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import HumanMessage, AnyMessage
from operator import add

# Shared state
class ChatState(TypedDict):
    messages: Annotated[list[AnyMessage], add]
    intent: str

# === SUBGRAPH: Intent Classification ===
def classify_intent(state: ChatState):
    """Subgraph node: Classify user intent"""
    last_message = state["messages"][-1].content.lower()
    
    if "name" in last_message:
        intent = "name_query"
    elif "weather" in last_message:
        intent = "weather_query"
    else:
        intent = "general"
    
    print(f"🔍 Subgraph - Intent: {intent}")
    return {"intent": intent}

def validate_intent(state: ChatState):
    """Subgraph node: Validate intent"""
    print(f"✅ Subgraph - Validated: {state['intent']}")
    return {}

# Build subgraph
def create_intent_subgraph():
    subgraph = StateGraph(ChatState)
    subgraph.add_node("classify", classify_intent)
    subgraph.add_node("validate", validate_intent)
    
    subgraph.add_edge(START, "classify")
    subgraph.add_edge("classify", "validate")
    subgraph.add_edge("validate", END)  # Subgraph has its own END
    
    return subgraph.compile()

# === PARENT GRAPH ===
def chatbot_node(state: ChatState):
    """Parent node: Generate response based on intent"""
    response = llm.invoke(state["messages"])
    print(f"🤖 Parent - Response generated")
    return {"messages": [response]}

def log_node(state: ChatState):
    """Parent node: Log conversation"""
    print(f"📝 Parent - Messages: {len(state['messages'])}, Intent: {state['intent']}")
    return {}

# Build parent graph
workflow = StateGraph(ChatState)

# Add subgraph as a node
workflow.add_node("intent_subgraph", create_intent_subgraph())
workflow.add_node("chatbot", chatbot_node)
workflow.add_node("logger", log_node)

# Add edges
workflow.add_edge(START, "intent_subgraph")  # Start with subgraph
workflow.add_edge("intent_subgraph", "chatbot")  # Subgraph END → chatbot
workflow.add_edge("chatbot", "logger")
workflow.add_edge("logger", END)  # Parent END

# Compile parent with checkpointer (auto-propagates to subgraph)
checkpointer = MemorySaver()
graph = workflow.compile(checkpointer=checkpointer)

print("=" * 50)
print("CONVERSATION 1")
print("=" * 50)

# First interaction
config = {"configurable": {"thread_id": "user-alice"}}
result1 = graph.invoke(
    {"messages": [HumanMessage(content="Hi! I'm Alice")]},
    config
)

print("\n" + "=" * 50)
print("CONVERSATION 2 (Remembers Alice)")
print("=" * 50)

# Second interaction - remembers Alice
result2 = graph.invoke(
    {"messages": [HumanMessage(content="What's my name?")]},
    config
)

print("\n" + "=" * 50)
print("FINAL STATE")
print("=" * 50)
print(f"Total messages: {len(result2['messages'])}")
print(f"Intent: {result2['intent']}")
print(f"Messages: {[m.content for m in result2['messages']]}")

# VectorDB Memory — Best Practices (Enterprise)

## Do You Save ALL Memories in VectorDB?

**No.** VectorDB is only ONE tier in a 3-tier memory stack.

In [ ]:
TIER 1 — Redis (Hot)         TIER 2 — PostgreSQL (Warm)    TIER 3 — VectorDB (Cold/Semantic)
─────────────────────────    ──────────────────────────    ─────────────────────────────────
Current session messages     Long-term facts & prefs       Semantic search over all memories
Last 5 messages in RAM       User profile, past episodes   Find relevant memories by meaning
< 1ms access                 2–5ms access                  20–50ms (ANN vector search)
TTL auto-expires             Explicit delete policy        Explicit delete policy

**Only semantically meaningful, long-term facts go to VectorDB** — not raw chat turns.

---

## What TO Save vs What NOT TO Save

In [ ]:
✅ SAVE IN VECTORDB                        ❌ DO NOT SAVE IN VECTORDB
──────────────────────────────────────     ──────────────────────────────────────
User preferences (language, tone)          Every chat message (use Redis instead)
Resolved complaints + outcomes             Transient intents ("check my balance")
Product interests + purchase history       Raw LLM responses
KYC / profile summaries                   Intermediate tool call results
Recurring issue patterns                   Duplicate / redundant facts

---

## VectorDB Has NO Built-in TTL — You Must Evict Explicitly

Unlike Redis (which auto-expires with TTL), VectorDB requires you to implement eviction.

### Pattern 1 — Metadata-Filtered Scheduled Eviction *(most common)*

In [ ]:
# Store metadata alongside every vector at write time
memory = {
    "text": "User prefers English responses",
    "metadata": {
        "user_id":          "u_123",
        "created_at":       "2026-01-01T00:00:00Z",
        "last_accessed_at": "2026-03-01T00:00:00Z",
        "access_count":     12,
        "memory_type":      "preference",   # preference | fact | episode | kyc
        "ttl_days":         180
    }
}

# Nightly cron job — evict stale memories
def evict_stale_memories(client, days_threshold=180):
    cutoff = datetime.utcnow() - timedelta(days=days_threshold)
    client.delete(filter={
        "last_accessed_at": {"$lt": cutoff.isoformat()},
        "memory_type":      {"$ne": "kyc"}   # never evict KYC (regulatory)
    })

### Pattern 2 — LRU-style Access Tracking

In [ ]:
# On every retrieval, refresh last_accessed_at to prevent eviction
def retrieve_and_refresh(query: str, user_id: str):
    results = vectordb.similarity_search(query, filter={"user_id": user_id}, k=5)
    for doc in results:
        vectordb.update(doc.id, metadata={"last_accessed_at": utcnow()})
    return results

### Pattern 3 — Tiered TTL by Memory Type

In [ ]:
Memory Type           TTL          Eviction Rule
────────────────────  ───────────  ──────────────────────────────────────
preference            365 days     Evict if not accessed in 1 year
complaint_resolved    90 days      Evict after resolution + 90 days
purchase_intent       30 days      Evict if no purchase in 30 days
kyc_profile           NEVER        Regulatory — archive to cold storage, never delete
session_summary       60 days      Evict after Redis session TTL expires

### Pattern 4 — Per-User Memory Cap

In [ ]:
# If a user exceeds N memories, evict the lowest-scoring ones
def enforce_memory_cap(user_id: str, cap: int = 100):
    memories = vectordb.fetch_all(filter={"user_id": user_id})
    if len(memories) > cap:
        scored = sorted(memories, key=lambda m: (
            m.metadata["access_count"] * 0.6 +
            recency_score(m.metadata["last_accessed_at"]) * 0.4
        ))
        evict_ids = [m.id for m in scored[:len(memories) - cap]]
        vectordb.delete(ids=evict_ids)

---

## Deduplication Before Insert

In [ ]:
def save_memory_with_dedup(user_id, content):
    query_vec = encoder.encode(content).tolist()
    results = vectordb.query(vector=query_vec, filter={"user_id": user_id}, top_k=1)

    # If very similar memory exists (cosine > 0.95), update instead of insert
    if results and results[0].score > 0.95:
        vectordb.update(results[0].id, metadata={"last_accessed_at": utcnow()})
        return

    # No duplicate — safe to insert
    vectordb.upsert([{
        "id":       f"{user_id}_{uuid.uuid4().hex}",
        "values":   query_vec,
        "metadata": {"user_id": user_id, "content": content,
                     "created_at": utcnow(), "last_accessed_at": utcnow(),
                     "access_count": 0, "memory_type": "preference"}
    }])

> Without dedup: 10k users × 100 duplicates = wasted cost + noisy retrieval results.

---

## Enterprise Rules

| Rule | Why |

|------|-----|

| **Write async** — never in request path | VectorDB upsert = 50–200ms; blocks LLM response |

| **Deduplicate** before insert (cosine > 0.95) | Prevents index bloat and noisy retrieval |

| **Namespace by user_id** | Never mix users in same namespace — data leak risk |

| **Never store raw PII** as vector text | Encrypt or tokenize before embedding (GDPR/DPDP) |

| **Archive KYC** — never delete | Regulatory obligation |

| **Cap memories per user** | Unbounded growth = slower ANN search over time |

---

## Scale Estimate (10M users)

In [ ]:
100 memories/user × 1536 dims × 4 bytes  ≈  600 MB raw vectors
+ metadata overhead                        ≈  1.5 GB total
Pinecone p1 pod handles                   ≈  5M vectors
Cost at 10M users                         ≈  $700–1400/month

VectorDB wins on cost at scale for long-term semantic memory.

Redis wins for session-scoped hot data.

---

## Summary

In [ ]:
SHORT-TERM (Redis)      → raw messages, TTL auto-expires, < 1ms
LONG-TERM (PostgreSQL)  → facts, prefs, episodes, explicit TTL
SEMANTIC  (VectorDB)    → only meaningful facts, evict with metadata filter + cron
                          always deduplicate, always namespace by user, always async write

---

# Isolating User Memory in VectorDB (Multi-User)

## The Problem

VectorDB uses **semantic similarity** — an unfiltered search can match

any user's memory, not just the current user's.

In [ ]:
User A asks: "What's my account balance limit?"
→ Without filter: may return User B's memory "balance limit is $5000"
→ With filter:    only searches within User A's memories  ✅

---

## Solution 1 — Always Filter by `user_id` on Every Query

Tag every memory with `user_id` at write time, and filter by it at read time.

In [ ]:
# WRITE — tag every memory with its owner
def save_memory(user_id: str, content: str):
    vector = encoder.encode(content).tolist()
    vectordb.upsert([{
        "id":     f"{user_id}_{uuid.uuid4().hex}",
        "values": vector,
        "metadata": {
            "user_id": user_id,   # ← owner tag
            "content": content,
        }
    }])

# READ — filter strictly to this user only
def get_memories(user_id: str, query: str):
    query_vec = encoder.encode(query).tolist()
    results = vectordb.query(
        vector=query_vec,
        filter={"user_id": {"$eq": user_id}},  # ← mandatory filter
        top_k=5,
        include_metadata=True
    )
    return [r["metadata"]["content"] for r in results["matches"]]

> **Rule: Never run a query without a `user_id` filter.** An unfiltered semantic search

> can match any user's memory.

---

## Solution 2 — Namespace Separation (Strongest Isolation)

Most VectorDBs support **namespaces** — physically separate partitions.

Even if the filter is accidentally omitted, namespaces prevent cross-user leaks.

In [ ]:
# WRITE — store in user's own namespace
index.upsert(vectors=[...], namespace=f"user_{user_id}")

# READ — search only within that namespace
index.query(vector=query_vec, namespace=f"user_{user_id}", top_k=5)

In [ ]:
Pinecone Index
├── namespace: user_101  →  only user 101's memories
├── namespace: user_202  →  only user 202's memories
└── namespace: user_303  →  only user 303's memories

No cross-contamination is possible — namespaces are physically isolated partitions.

---

## Solution 3 — Always Derive `user_id` from Auth Token

Never trust a `user_id` supplied by the client — always extract it from

the authenticated session token server-side.

In [ ]:
def get_memories(request: Request, query: str):
    user_id = get_user_from_jwt(request.headers["Authorization"])  # from token
    if not user_id:
        raise PermissionError("Unauthenticated")

    return vectordb.query(filter={"user_id": user_id}, ...)

---

## Enterprise: Add Tenant-Level Isolation (Multi-Company SaaS)

In [ ]:
# Filter by both company AND user — two layers of isolation
results = vectordb.query(
    vector=query_vec,
    filter={
        "tenant_id": "company_A",   # company-level
        "user_id":   "user_101"     # user-level
    },
    top_k=5
)

In [ ]:
LAYER 1 — Namespace      : per tenant (company)
LAYER 2 — Metadata filter: per user within tenant
LAYER 3 — Auth check     : verify user belongs to tenant

---

## Summary

| Layer | What It Does |

|-------|-------------|

| `user_id` in metadata | Tags every memory with its owner at write time |

| `filter={"user_id": ...}` on query | Restricts ANN search to that user only |

| Namespace per user/tenant | Physical partition — strongest isolation |

| Auth token validation | Never trust client-supplied `user_id` |

| Tenant filter | Adds company-level isolation for SaaS |